In [ ]:
# RAG pipeline — Groq API only, zero torch, zero sentence-transformers
import os, json
from openai import OpenAI
from dotenv import load_dotenv

env_path = r"C:\Users\User\Desktop\micronprep\.env"
load_dotenv(env_path)

client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

# ── 1. Document store (simulates indexed spec docs) ────────────────
docs = [
    {"text": "NVMe supports up to 65535 I/O queues with 65535 commands per queue, enabling massive flash parallelism.", "source": "NVMe_2.0_spec", "section": "3.1 Queue Model"},
    {"text": "The Flash Translation Layer (FTL) maps logical block addresses to physical NAND, handles wear levelling, GC, and bad blocks.", "source": "SSD_Architecture_Guide", "section": "FTL Overview"},
    {"text": "PCIe 4.0 gives ~2 GB/s per lane. NVMe SSDs use x4 for 8 GB/s total bandwidth.", "source": "PCIe_4.0_spec", "section": "Bandwidth"},
    {"text": "UFS 3.1 supports full-duplex at 23.2 Gbps per lane over M-PHY physical layer.", "source": "JEDEC_UFS_3.1", "section": "Physical Layer"},
    {"text": "ECC (Error Correcting Code) detects and corrects bit errors in NAND flash. Uncorrectable ECC errors indicate block failure.", "source": "NAND_Flash_Guide", "section": "ECC"},
]

# ── 2. Simple keyword retrieval (no FAISS needed for demo) ─────────
def retrieve(query: str, top_k: int = 2) -> list:
    query_words = set(query.lower().split())
    scored = []
    for doc in docs:
        doc_words = set(doc["text"].lower().split())
        score = len(query_words & doc_words)
        scored.append((score, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [d for _, d in scored[:top_k]]

# ── 3. RAG: retrieve + generate ────────────────────────────────────
query = "How many queues does NVMe support?"
retrieved = retrieve(query)

print(f"Query: {query}")
print("\nRetrieved chunks:")
for i, doc in enumerate(retrieved):
    print(f"  [{i+1}] {doc['source']} — {doc['section']}")
    print(f"       {doc['text']}")

context = "\n\n".join([f"[{d['source']}]: {d['text']}" for d in retrieved])

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "Answer ONLY from the provided context. If not found, say 'Not in indexed documents.'"},
        {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {query}"}
    ],
    temperature=0.0,
    max_tokens=200
)

print(f"\nFinal Answer:\n{response.choices[0].message.content}")
print(f"\nSources: {[d['source'] for d in retrieved]}")

Query: How many queues does NVMe support?

Retrieved chunks:
  [1] NVMe_2.0_spec — 3.1 Queue Model
       NVMe supports up to 65535 I/O queues with 65535 commands per queue, enabling massive flash parallelism.
  [2] PCIe_4.0_spec — Bandwidth
       PCIe 4.0 gives ~2 GB/s per lane. NVMe SSDs use x4 for 8 GB/s total bandwidth.


NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama3.2:latest` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}